# Module 06 — Tracing & Inspection *(interactive)*

> Optimizers are black boxes until you **crack them open.** This notebook loads the
> *real* compiled programs and scorecards we produced and renders them inline — the demos
> a few-shot optimizer chose, the teacher's reasoning traces, the 8,078-character instruction
> GEPA evolved, and the full before→after portfolio.

**Runs anywhere — no GPU, no LM calls, no API key.** Every cell reads artifacts already on
disk and draws them with pure HTML/CSS. Run the cells top to bottom, then edit the last one to
inspect any program yourself.

---

## 1 · Setup — load artifacts & define the look

These helpers render self-contained dark panels that look good on any notebook theme.

In [1]:
import json
from pathlib import Path
from IPython.display import HTML, display

# Find the project root (the folder that contains scorecards/) by walking up.
ROOT = Path.cwd()
while not (ROOT / "scorecards").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project root:", ROOT)

def load(p):
    return json.loads((ROOT / p).read_text(encoding="utf-8"))

PAL = dict(line="#1f2c45", tx="#e2e8f0", mut="#94a3b8",
           cy="#22d3ee", em="#34d399", vi="#a78bfa", am="#fbbf24")

def panel(inner, pad=18):
    return (f'<div style="background:linear-gradient(180deg,#111a2e,#0f1729);color:{PAL["tx"]};'
            f'border:1px solid {PAL["line"]};border-radius:14px;padding:{pad}px;'
            f'font-family:ui-sans-serif,system-ui,Segoe UI,Roboto,sans-serif;line-height:1.55">{inner}</div>')

def scroll(text, h=200):
    return (f'<div style="font-size:12.5px;color:#c8d4e6;white-space:pre-wrap;max-height:{h}px;'
            f'overflow:auto;border:1px solid {PAL["line"]};border-radius:8px;padding:10px">{text}</div>')

def show(inner):  # render a panel
    display(HTML(panel(inner)))

print("Helpers ready: load(), panel(), scroll(), show()")

Project root: e:\miniconda\envs\optimize\files
Helpers ready: load(), panel(), scroll(), show()


## 2 · What *is* a compiled program?

A DSPy optimizer outputs a JSON program: a predictor with its **instruction** (signature) and its **demonstrations** (few-shot examples). Let's look at the shape.

In [2]:
prog = load("compiled/ticket_router/labeled_fewshot.json")
pred_key = next(k for k in prog if k != "metadata")     # e.g. "classify.predict"
sig   = prog[pred_key].get("signature", {})
demos = prog[pred_key].get("demos", [])

show(f'<b>Predictor:</b> <code style="color:{PAL["cy"]}">{pred_key}</code>'
     f'<br><b>Instruction:</b> {len(sig.get("instructions",""))} chars'
     f'<br><b>Demonstrations:</b> {len(demos)} examples'
     f'<br><b>Demo fields:</b> {", ".join(demos[0].keys()) if demos else "—"}')

## 3 · The demos LabeledFewShot selected

Few-shot optimization is just *picking the most useful examples* to show the model. Here are the actual tickets it chose for UC01:

In [3]:
def ticket_card(d):
    return (f'<div style="border:1px solid {PAL["line"]};border-radius:10px;padding:11px 13px;'
            f'margin:8px 0;background:#0c1730">'
            f'<div style="font-size:11px;color:{PAL["mut"]};letter-spacing:.06em">TICKET</div>'
            f'<div style="font-weight:600;margin:2px 0 5px">{d.get("subject","")}</div>'
            f'<div style="font-size:12.5px;color:#c8d4e6">{(d.get("body","") or "")[:170]}…</div>'
            f'<div style="margin-top:9px">'
            f'<span style="background:#0e3a2a;color:{PAL["em"]};padding:3px 9px;border-radius:6px;font-size:12px">{d.get("department","?")}</span> '
            f'<span style="background:#3a2a0e;color:{PAL["am"]};padding:3px 9px;border-radius:6px;font-size:12px">priority: {d.get("priority","?")}</span></div></div>')

show(f'<div style="font-weight:700;margin-bottom:4px">3 of the {len(demos)} demonstrations '
     f'the optimizer attached</div>' + "".join(ticket_card(d) for d in demos[:3]))

## 4 · Bootstrapped demos carry the teacher's *reasoning*

`BootstrapFewShot` is knowledge distillation: the strong teacher works an example step-by-step, and that reasoning trace becomes a demo. Here's a real one from the contract extractor (UC02, the +10.6% winner):

In [4]:
demos2 = load("compiled/contract_extractor/bootstrap_rs.json")["extract.predict"]["demos"]
aug = [d for d in demos2 if d.get("augmented")] or demos2
d = aug[0]

inner = '<div style="font-weight:700;margin-bottom:6px">A teacher-bootstrapped extraction</div>'
inner += f'<div style="font-size:11px;color:{PAL["mut"]};letter-spacing:.06em">REASONING TRACE</div>'
inner += scroll(d.get("reasoning") or "(this demo had no reasoning trace)", h=150)
inner += f'<div style="font-size:11px;color:{PAL["mut"]};letter-spacing:.06em;margin-top:10px">EXTRACTED FIELDS</div>'
for k in ["parties", "effective_date", "total_value", "payment_terms", "governing_law"]:
    if d.get(k):
        inner += f'<div style="font-size:12.5px;margin:2px 0"><b style="color:{PAL["cy"]}">{k}:</b> {d[k]}</div>'
show(inner)

## 5 · 🌟 The instruction GEPA *wrote* — 190 → 8,078 chars

This is the headline artifact. Blind optimizers plateaued at +2% on the clinical note; GEPA read per-field feedback and **rewrote the instruction**, growing it from a human's ~190 characters to over 8,000 — and it contains the exact `follow_up` formatting rule the blind search never found.

In [5]:
original = ("Summarize patient encounter note into structured clinical fields. Use medical "
            "knowledge to interpret abbreviations. Extract all relevant diagnoses, planned "
            "actions, and medication changes.")   # from program.py (the human-written signature)
evolved = load("compiled/clinical_note/gepa_feedback.json")["summarize.predict"]["signature"]["instructions"]

def col_block(label, text, color):
    return (f'<div style="flex:1;min-width:0">'
            f'<div style="font-size:11px;color:{color};font-weight:700;text-transform:uppercase;'
            f'letter-spacing:.08em">{label} · {len(text):,} chars</div>{scroll(text, h=300)}</div>')

show(f'<div style="font-weight:700;margin-bottom:8px">Before (human) vs after (GEPA reflective evolution)</div>'
     f'<div style="display:flex;gap:12px;flex-wrap:wrap">'
     f'{col_block("Original", original, PAL["mut"])}'
     f'{col_block("GEPA-evolved", evolved, PAL["vi"])}</div>')

## 6 · The whole portfolio — before → after

Five scorecards, drawn as dashboard-style bars (dashed line = the baseline each optimizer had to beat). This is the same data as `scorecards/portfolio.html`, rendered live from the JSONs.

In [6]:
UCS = [("ticket_router","UC01 Ticket Router"),("contract_extractor","UC02 Contract Extractor"),
       ("rag_answer","UC03 RAG Answer"),("clinical_note","UC04 Clinical Note"),
       ("sales_email","UC05 Sales Email")]

rows = '<div style="font-weight:700;margin-bottom:10px">Portfolio — baseline → best ' \
       '<span style="color:#93a4c4;font-weight:400;font-size:12px">(dashed = baseline)</span></div>'
for f, name in UCS:
    s = load(f"scorecards/{f}.json")["summary"]
    base, best, delta = s["baseline_score"], s["best_score"], s["headline_delta"]
    win = delta > 0.001
    col = PAL["em"] if win else PAL["am"]
    badge = ("+%.1f%%" % (delta*100)) if win else "+0.0%"
    rows += (f'<div style="margin:11px 0">'
             f'<div style="display:flex;justify-content:space-between;font-size:13px">'
             f'<b>{name}</b><span style="color:{col};font-weight:800">{badge}</span></div>'
             f'<div style="position:relative;height:11px;background:#0c1426;border:1px solid #16223c;'
             f'border-radius:6px;margin-top:5px">'
             f'<div style="position:absolute;height:100%;width:{best*100:.1f}%;border-radius:6px;'
             f'background:linear-gradient(90deg,#0e7490,{col})"></div>'
             f'<div style="position:absolute;left:{base*100:.1f}%;top:-2px;bottom:-2px;'
             f'border-left:2px dashed #93a4c4;opacity:.75"></div></div>'
             f'<div style="font-size:11px;color:{PAL["mut"]};margin-top:3px">'
             f'baseline {base*100:.1f}% → best {best*100:.1f}% &middot; {s["best_label"]}</div></div>')
show(rows)

> **Reminder:** these percentages use *different metrics per task* — read the **lift (the badge)**, not the absolute bar length. UC05's +11.4% is the biggest win despite the lowest bar.

## 7 · Inspect any program yourself

One reusable function. **Change the arguments and re-run** to crack open any compiled program in the repo.

In [7]:
def inspect(use_case, label):
    prog = load(f"compiled/{use_case}/{label}.json")
    pred = next(k for k in prog if k != "metadata")
    sig  = prog[pred].get("signature", {})
    demos = prog[pred].get("demos", [])
    instr = sig.get("instructions", "")
    inner = (f'<div style="font-weight:700">{use_case} · {label}</div>'
             f'<div style="font-size:12px;color:{PAL["mut"]};margin:6px 0">'
             f'{len(demos)} demos &middot; instruction {len(instr):,} chars</div>'
             f'{scroll(instr or "(no custom instruction — uses the original signature)", h=220)}')
    show(inner)

# Available programs:
for uc in ["ticket_router","contract_extractor","rag_answer","clinical_note","sales_email"]:
    progs = sorted(p.stem for p in (ROOT/"compiled"/uc).glob("*.json"))
    print(f"{uc:20s}: {progs}")

print("\nTry it:")
inspect("sales_email", "gepa_feedback")   # ← change these and re-run

ticket_router       : ['bootstrap', 'bootstrap_rs', 'bootstrap_rs_cot', 'labeled_fewshot', 'labeled_fewshot_cot', 'mipro_light', 'mipro_medium', 'mipro_medium_cot']
contract_extractor  : ['bootstrap_rs', 'gepa', 'labeled_fewshot', 'mipro_medium']
rag_answer          : ['bootstrap_rs', 'labeled_fewshot', 'mipro_medium']
clinical_note       : ['copro_instructions', 'gepa_feedback', 'mipro_instructions']
sales_email         : ['bootstrap_rs', 'gepa_feedback', 'labeled_fewshot', 'mipro_light']

Try it:


## 8 · What you just did

```
┌────────────────────────────────────────────────────────────────┐
│ 1. A compiled program = an instruction + demonstrations, in JSON.│
│ 2. Few-shot optimizers SELECT demos; bootstrap GENERATES them    │
│    with the teacher's reasoning attached.                        │
│ 3. GEPA evolves the INSTRUCTION itself — here, 190 → 8,078 chars.│
│ 4. Every result is inspectable. Never trust a score you can't    │
│    open up and explain.                                          │
└────────────────────────────────────────────────────────────────┘
```

**Companion reading:** [Module 06 (markdown)](../06_tracing_and_inspection.md) ·
[Module 04b — GEPA](../04b_gepa.md) · the live dashboard `scorecards/portfolio.html`.